# ⚡ MeatVision AI — High-Speed GPU Model Training (Google Colab)

This notebook trains both **Meat Species Classification** (Beef, Chicken, Fish, Pork) and **Meat Freshness Detection** (Fresh, Half Fresh, Spoiled) using Google Colab T4 GPU acceleration.

### 🚀 Quick Start Instructions:
1. Click **Runtime** $\rightarrow$ **Change runtime type** $\rightarrow$ Select **T4 GPU**.
2. Execute Cell 1 to set up PyTorch & project files.
3. Execute Cell 2 & Cell 3 to train both models in **~3 minutes total**.
4. Download the resulting `species_model.pth` and `freshness_model.pth` checkpoints back into your local `models/` directory.

### Step 1: Check GPU & Setup Environment

In [5]:
import torch
import os
from pathlib import Path

print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
else:
    print('⚠️ WARNING: No GPU detected. Please go to Runtime -> Change runtime type -> T4 GPU.')


CUDA Available: False
⚠️ WARNING: No GPU detected. Please go to Runtime -> Change runtime type -> T4 GPU.


### Step 2: Upload or Clone Dataset & Codebase
*(Upload `MeatVisionAI.zip` or mount Google Drive if dataset is in Drive)*

In [6]:
# Step 2: Setup Project Code, Dataset & Mount Drive
import os, sys, zipfile, subprocess
from pathlib import Path

# 1. Mount Google Drive (Optional)
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        try:
            drive.mount('/content/drive')
            print('✅ Google Drive mounted.')
        except Exception:
            print('Google Drive mount skipped.')
except ImportError:
    print('Not running inside Google Colab environment.')

# 2. Locate or Prepare Codebase
possible_roots = [
    Path('/content/MeatVisionAI'),
    Path('/content/MeatVision_Project'),
    Path('/content/MeatVision_Code'),
    Path('/content')
]

project_root = None
for p in possible_roots:
    if (p / 'scripts' / 'training' / 'train_species.py').exists():
        project_root = p
        break

if not project_root:
    repo_zips = [z for z in Path('/content').glob('*.zip') if 'dataset' not in z.name.lower()]
    for zip_path in repo_zips:
        print(f'📦 Extracting project zip: {zip_path.name}...')
        target = Path('/content/MeatVisionAI')
        target.mkdir(exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(target)
        if (target / 'scripts' / 'training' / 'train_species.py').exists():
            project_root = target
            break

if not project_root:
    print('🔄 Zip file not found in /content. Auto-cloning MeatVisionAI from GitHub...')
    if Path('/content/MeatVisionAI').exists():
        subprocess.run(['git', '-C', '/content/MeatVisionAI', 'pull'], check=False)
    else:
        subprocess.run(['git', 'clone', 'https://github.com/JoelJames889/MeatVisionAI.git', '/content/MeatVisionAI'], check=False)
    
    if (Path('/content/MeatVisionAI') / 'scripts' / 'training' / 'train_species.py').exists():
        project_root = Path('/content/MeatVisionAI')

if project_root:
    print(f'✅ MeatVision AI codebase ready at: {project_root}')
    os.chdir(project_root)
    sys.path.insert(0, str(project_root / 'scripts' / 'training'))
    print(f'📂 Active directory set to: {os.getcwd()}')

# 3. Locate & Extract Dataset if uploaded as zip
dataset_zips = list(Path('/content').glob('*dataset*.zip')) + list(Path('/content').glob('*Dataset*.zip'))
if dataset_zips and project_root:
    for dz in dataset_zips:
        print(f'📦 Extracting Dataset zip: {dz.name}...')
        with zipfile.ZipFile(dz, 'r') as z:
            z.extractall(project_root)

# Check for species dataset directory
species_found = False
check_paths = [
    (project_root / 'Dataset' / 'species') if project_root else None,
    Path('/content/drive/MyDrive/Dataset/species'),
    Path('/content/drive/MyDrive/MeatVisionAI/Dataset/species'),
    Path('/content/Dataset/species')
]

for cp in check_paths:
    if cp and cp.exists() and any(cp.iterdir()):
        species_found = True
        print(f'✅ Dataset verified at: {cp}')
        break

if not species_found:
    print('\n' + '='*80)
    print('⚠️ NOTICE: DATASET IMAGES NOT FOUND YET')
    print('='*80)
    print('To run training, please ensure your dataset images are available in Colab:')
    print('  Option A: Mount Google Drive if your dataset is in MyDrive/Dataset/')
    print('  Option B: Upload your Dataset folder (or Dataset.zip) to /content/ and extract it:')
    print('            !unzip /content/Dataset.zip -d /content/MeatVisionAI/')
    print('='*80 + '\n')


### Step 3: Train Species Classification Model (`species_model.pth`)

In [ ]:
# Execute Species Training Script
!python scripts/training/train_species.py


### Step 4: Train Freshness Detection Model (`freshness_model.pth`)

In [ ]:
# Execute Freshness Training Script
!python scripts/training/train_freshness.py


### Step 5: Download Trained Checkpoints to Your Machine

In [ ]:
from google.colab import files
import os

models_to_download = ['models/species_model.pth', 'models/freshness_model.pth']
for m in models_to_download:
    if os.path.exists(m):
        print(f'Downloading {m}...')
        files.download(m)
    else:
        print(f'Checkpoint {m} not found.')
